# PMIL — Camelyon16 Data Prep Pipeline

This notebook mounts Google Drive, sets up the expected folder structure, **downloads the Camelyon16 training WSIs from the public AWS S3 mirror** (no AWS account needed), installs the `pmil` package, generates a labels CSV, and runs tiling/feature-prep.

Source: https://registry.opendata.aws/camelyon/ — bucket `s3://camelyon-dataset` (region `us-west-2`, public, `--no-sign-request`).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set paths and create folder structure

In [ ]:
import os

WSI_DIR = '/content/drive/MyDrive/camelyon16/training'
LABELS_CSV = '/content/drive/MyDrive/camelyon16/train_labels.csv'
DRIVE_OUT = '/content/drive/MyDrive/pmil'

os.makedirs(WSI_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_OUT}/features/train", exist_ok=True)
os.makedirs(f"{DRIVE_OUT}/features/val", exist_ok=True)
os.makedirs(f"{DRIVE_OUT}/artifacts", exist_ok=True)
SPLITS_DIR = f"{DRIVE_OUT}/splits"
os.makedirs(SPLITS_DIR, exist_ok=True)
os.environ['SPLITS_DIR'] = SPLITS_DIR

print("WSI_DIR:", WSI_DIR)
print("LABELS_CSV:", LABELS_CSV)
print("DRIVE_OUT:", DRIVE_OUT)

os.environ['WSI_DIR'] = WSI_DIR
os.environ['LABELS_CSV'] = LABELS_CSV
os.environ['DRIVE_OUT'] = DRIVE_OUT

## 3. Install the AWS CLI and explore the S3 bucket (do this first, before downloading anything)
`--no-sign-request` means no AWS credentials are needed since the bucket is public. Run the listing below and confirm the CAMELYON16 training path before syncing — bucket layouts on these mirrors sometimes shift, so it's worth eyeballing it once rather than assuming.

In [ ]:
!pip install awscli --quiet

In [ ]:
# Top-level layout
!aws s3 ls --no-sign-request s3://camelyon-dataset/

In [ ]:
# Drill into CAMELYON16 (adjust the path below based on what step 3's top-level listing showed)
!aws s3 ls --no-sign-request s3://camelyon-dataset/CAMELYON16/

In [ ]:
# The WSIs live under images/ (masks/ and annotations/ are separate folders).
# NOTE: per community-documented usage, images/ mixes training slides (tumor_*.tif, normal_*.tif)
# and test slides (test_*.tif) together in one folder -- confirm this with your own eyes below
# before assuming the pattern, since bucket layouts can change.
!aws s3 ls --no-sign-request s3://camelyon-dataset/CAMELYON16/images/ | head -20

## 4. Download the training WSIs into Drive

**Read this before running:**

- **This step assumes step 3's listing confirmed the `tumor_*` / `normal_*` / `test_*` naming pattern.** If the actual filenames look different, fix the `--include` patterns below to match before running a multi-hour sync.
- **Full CAMELYON16 training set (tumor+normal) is several hundred GB.** For a first pipeline test, consider downloading just a handful of files first (see the single-file test cell below) rather than the full `sync`.
- **Drive storage.** Make sure your Google Drive quota can hold this (check drive.google.com/settings/storage). Colab's local disk (`/content`) is faster to write to than Drive but usually smaller and wiped when the session ends — you may prefer downloading raw WSIs locally and only pushing the tiles/features (much smaller) to Drive afterward.
- **Session time limits / resumability.** Colab sessions can disconnect mid-download. `aws s3 sync` skips files it already has, so if it cuts off, just rerun the same command.

**Test with a single file first:**

In [ ]:
# Sanity check: pull one known file first. Replace with an actual filename you saw in step 3's listing.
!aws s3 cp --no-sign-request s3://camelyon-dataset/CAMELYON16/annotations/tumor_032.xml /tmp/test_download.xml
!ls -la /tmp/test_download.xml

**Full training sync**, once the single-file test above works. This pulls only training slides (`tumor_*`, `normal_*`) and skips the `test_*` files, based on the naming pattern check in step 3 — adjust the `--include` patterns if the real listing looked different:

**Important: do not interrupt this halfway.** Last time this got `^C`'d after only `normal_*` files finished, which is exactly what broke every step downstream (labels generation needs both classes present). If you want a smaller download you won't be tempted to cut off, use the **subset option** below instead of the full sync.

### Recommended for a first pipeline run: small subset (a few minutes, not hours)
Grabs just the first few tumor and normal slides — enough for `make_camelyon16_labels` to see both classes and for the whole pipeline to run end-to-end quickly. Increase `N_PER_CLASS` later once the pipeline is verified working.

In [ ]:
N_PER_CLASS = 5  # slides per class to download for the smoke test; raise later for a real training run

import subprocess

def list_slides(prefix, n):
    out = subprocess.run(
        ['aws', 's3', 'ls', '--no-sign-request', f's3://camelyon-dataset/CAMELYON16/images/{prefix}'],
        capture_output=True, text=True, check=True
    ).stdout
    names = [line.split()[-1] for line in out.strip().splitlines() if line.strip()]
    return sorted(names)[:n]

subset = list_slides('normal_', N_PER_CLASS) + list_slides('tumor_', N_PER_CLASS)
print(f"Downloading {len(subset)} slides:", subset)

for name in subset:
    dest = f"{WSI_DIR}/{name}"
    if os.path.isfile(dest):
        print(f"already have {name}, skipping")
        continue
    subprocess.run(
        ['aws', 's3', 'cp', '--no-sign-request',
         f's3://camelyon-dataset/CAMELYON16/images/{name}', dest],
        check=True
    )
print("Done.")

### Full training sync (hundreds of GB — only do this once the subset run above has verified the pipeline works)
This pulls the entire training set. Let it run uninterrupted to the end, or if you do need to stop it, rerun the same command later — `aws s3 sync` skips files it already has.

In [ ]:
!aws s3 sync --no-sign-request s3://camelyon-dataset/CAMELYON16/images/ "$WSI_DIR" \
  --exclude "*" \
  --include "tumor_*.tif" \
  --include "normal_*.tif"

## 5. Confirm the download landed correctly

In [ ]:
wsi_files = [f for f in os.listdir(WSI_DIR) if f.lower().endswith(('.tif', '.tiff', '.svs'))]
print(f"Found {len(wsi_files)} WSI file(s) in {WSI_DIR}")
if wsi_files:
    print("Sample:", wsi_files[:5])
else:
    print("⚠️  Still no WSI files — re-check the S3 path used in step 4.")

## 6. Install the `pmil` package (from your uploaded zip, stored persistently in Drive)

`/content` gets wiped every time a Colab session ends — that's why the repo disappeared last time. This step unzips it into your **Drive** instead, so you only need to upload it once, ever. If `pmil-colab-reproduction/` already exists in Drive (e.g. from a previous run), this skips the upload and reuses it.

In [ ]:
import os, zipfile

REPO_DIR = '/content/drive/MyDrive/pmil-colab-reproduction'
os.environ['REPO_DIR'] = REPO_DIR

if os.path.isdir(REPO_DIR) and os.path.isfile(os.path.join(REPO_DIR, 'pyproject.toml')):
    print(f"Found existing repo at {REPO_DIR}, skipping upload.")
else:
    from google.colab import files
    print("Upload your pmil-colab-reproduction zip file now:")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    extract_to = '/content/drive/MyDrive'
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(extract_to)
    print(f"Extracted to {extract_to}")
    assert os.path.isfile(os.path.join(REPO_DIR, 'pyproject.toml')), (
        f"pyproject.toml not found at {REPO_DIR} -- check the zip's internal folder name matches REPO_DIR"
    )

In [ ]:
# System library required by openslide-python (prepare_camelyon16.py imports openslide)
!apt-get install -y -qq libopenslide0
!pip install -e "$REPO_DIR" --quiet

## 7. Generate the labels CSV from the WSI folder

In [ ]:
!python -m pmil.make_camelyon16_labels \
  --wsi-dir "$WSI_DIR" \
  --output "$LABELS_CSV"

## 8. Prepare tiles / splits for training

In [ ]:
!python -m pmil.prepare_camelyon16 \
  --wsi-dir "$WSI_DIR" \
  --labels-csv "$LABELS_CSV" \
  --output-dir "$SPLITS_DIR" \
  --level 1 --tile-size 256 --stride 256 \
  --min-tissue-fraction 0.05 --seed 24

## 9. Confirm GPU is available
The next stage trains a ResNet-34 while streaming patches from your WSI files. On CPU this is impractically slow. If this cell prints `False`, go to Runtime -> Change runtime type -> select a GPU (T4 is fine), then rerun from the top.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 10. Train the Max-MIL instance encoder (+ extract train feature bags)

**Strongly recommended: do a smoke test first.** The full run below defaults to 30 epochs, which can take a long time given only a handful of slides are downloaded so far. Before committing to a long run, try `--epochs 2` to confirm the whole command executes cleanly end-to-end (correct paths, openslide reads working, checkpoint gets written), then rerun with more epochs once that succeeds.

This step both trains the encoder AND writes extracted feature bags for the **train** split to `$DRIVE_OUT/features/train`.

In [ ]:
!python -m pmil.train_maxmil \
  --slides-csv "$SPLITS_DIR/train.csv" \
  --val-csv "$SPLITS_DIR/val.csv" \
  --patch-root "$WSI_DIR" \
  --checkpoint "$DRIVE_OUT/artifacts/maxmil.pt" \
  --feature-dir "$DRIVE_OUT/features/train" \
  --epochs 2

Once the smoke test above succeeds, rerun with a real epoch count (uncomment and edit as needed):

In [ ]:
# !python -m pmil.train_maxmil \
#   --slides-csv "$SPLITS_DIR/train.csv" \
#   --val-csv "$SPLITS_DIR/val.csv" \
#   --patch-root "$WSI_DIR" \
#   --checkpoint "$DRIVE_OUT/artifacts/maxmil.pt" \
#   --feature-dir "$DRIVE_OUT/features/train" \
#   --epochs 30

## 11. Extract feature bags for the validation split
`train_maxmil` only writes train-split features. This uses the checkpoint it just produced to extract val-split features the same way.

In [ ]:
!python -m pmil.extract_features \
  --slides-csv "$SPLITS_DIR/val.csv" \
  --patch-root "$WSI_DIR" \
  --checkpoint "$DRIVE_OUT/artifacts/maxmil.pt" \
  --feature-dir "$DRIVE_OUT/features/val"

## 12. Cluster prototypes from the train feature bags

In [ ]:
!python -m pmil.cluster_prototypes \
  --slides-csv "$SPLITS_DIR/train.csv" \
  --feature-dir "$DRIVE_OUT/features/train" \
  --output "$DRIVE_OUT/artifacts/prototypes.npy"

## 13. Train the final PMIL model
This is the model checkpoint your run is ultimately producing.

In [ ]:
!python -m pmil.train_pmil \
  --train-csv "$SPLITS_DIR/train.csv" \
  --val-csv "$SPLITS_DIR/val.csv" \
  --train-features "$DRIVE_OUT/features/train" \
  --val-features "$DRIVE_OUT/features/val" \
  --prototypes "$DRIVE_OUT/artifacts/prototypes.npy" \
  --instance-checkpoint "$DRIVE_OUT/artifacts/maxmil.pt" \
  --checkpoint "$DRIVE_OUT/artifacts/pmil_final.pt" \
  --epochs 30

## 14. Confirm the final model was produced

In [ ]:
import os
final_ckpt = f"{DRIVE_OUT}/artifacts/pmil_final.pt"
if os.path.isfile(final_ckpt):
    size_mb = os.path.getsize(final_ckpt) / 1e6
    print(f"Model saved: {final_ckpt} ({size_mb:.1f} MB)")
else:
    print("No checkpoint found yet -- check the training cell above for errors.")